# Fit time-dependent estendido com produção, tags e background

Exemplo compacto da composição `production_fraction -> signal_tag_fraction`, yields estendidos e background Dalitz-tempo fatorizado. As formas de background são artificiais.

In [1]:
import jax.numpy as jnp
from dalitzplotfitter import (
    DecayChannel, DecayModel, NeutralMesonMixing, NonResonant, Parameter,
    PhaseSpaceSample, RealImag, TimeDependentFitSession,
)

channel = DecayChannel("D0", ("K(S)0", "pi+", "pi-"))
model = DecayModel(
    channel, [NonResonant(RealImag(1.0, 0.0))],
    normalization_method="square-dalitz", normalization_resolution=16,
)
data = model.generate_phase_space(2_000, seed=4, include_momenta=False)
times = jnp.linspace(0.01, 3.99, data.size)
tags = jnp.where(jnp.arange(data.size) % 2 == 0, 1, -1)

def efficiency(events):
    return jnp.clip(0.8 + 0.08 * jnp.sin(events["s12"] + events["s13"]), 0.3, 1.0)

def background_shape(events):
    return 1.0 + 0.2 * jnp.cos(events["s12"])

def background_time(events, values):
    rate = 0.7
    norm = 1.0 - jnp.exp(-rate * 4.0)
    return rate * jnp.exp(-rate * events["t"]) / norm

In [2]:
session = TimeDependentFitSession(
    model, data, times, tags, NeutralMesonMixing(0.01, 0.005, 0.4103),
    efficiency=efficiency, time_range=(0.0, 4.0),
    wrong_tag=0.08, extended=True,
    signal_yield=Parameter("N_signal", 1_600.0, bounds=(0.0, None)),
    production_fraction=Parameter("f_production", 0.65, bounds=(0.0, 1.0)),
).with_background(
    "combinatorial", background_shape, time_pdf=background_time,
    yield_=Parameter("N_background", 400.0, bounds=(0.0, None)),
    tag_fraction=0.5,
)

result = session.fit(strategy=1, hesse=False, ncall=100)
session.print_result(result)

valid=False  NLL=-6058.301543
parameter                           value            error
N_background                         2000                0
N_signal                      1.03062e-09                0
f_production                     0.381068                0


{'N_background': 1999.999999947638,
 'N_signal': 1.0306189130197471e-09,
 'f_production': 0.38106776659018965}

`production_fraction` é a fração verdadeira de D0. A sessão converte-a para a fração observada de tags usando `wrong_tag`:

`P(tag=+1) = f_production (1-w) + (1-f_production) w`.

Para dados reais, os parâmetros de produção e tag devem ser calibrados por categoria e podem receber constraints externos.